# Deep-Mechinterp Starter: AtomicDance Planner — v2 (reviewt & gefixt)

Aufbau wie gehabt: Stimulus-Suite → Aktivierungs-Hooks → Kontrastanalyse → lineare Probes → D3PM-Trajektorie → Kausal-Skelett. Das echte, selbst trainierte Planner-Modell wird **nur lesend** von Drive geladen; der Stimulus ist Musik (Faktoren aus den AIST++-Sequenznamen).

**Änderungen in v2 (nach Review):**

1. **Modell-Rekonstruktion gefixt (Abschnitt 5).** Die Auto-Discovery ist durch die verifizierte direkte Konstruktion ersetzt: `UniformD3PM(AtomicPlannerTransformer(num_atomic_classes=100, music_dim=35, max_seq_len=150))`. Im Quellcode gilt `num_classes = num_atomic_classes + 1` → Checkpoint-Embedding `[101, 512]` ⇒ **K=100**. `max_seq_len=150` ist zwingend (Klassen-Default 18000 ergäbe einen `pe`-Buffer `[18000, 1, 512]` → Size-Mismatch). Lokal gegen den Checkpoint-Abdruck bestätigt: 110 Tensoren, 19.121.965 Elemente, `strict=True` lädt.
   *Nebenbefund: `load_state_dict(strict=False)` wirft bei Size-Mismatch trotzdem `RuntimeError` — daran starb die alte K-Schleife, obwohl K=100 vermutlich schon erfolgreich geladen hatte.*
2. **Parameterdiskrepanz erklärt:** 19.121.965 (State-Dict) = 19.044.965 trainierbar + 76.800 (`pe [150, 1, 512]`) + 200 (`alphas` + `alpha_bars`). Kein Problem — Buffer vs. Parameter.
3. **Probe-Split gefixt:** `GroupShuffleSplit` gruppiert jetzt nach **Musik-ID** statt `base_seq`. Der Probe-Input ist ausschließlich Musik, und derselbe Song wird in AIST++ von mehreren Tänzern getanzt — sonst landet identischer Input in Train **und** Test (Leakage).
4. **Situations-Probe = Negativkontrolle:** sBM/sFM teilen dieselben Songs; erwartet ist Chance-Niveau. Deutlich mehr ⇒ Leakage-/Artefakt-Alarm.
5. **Trajektorie über mehrere Sequenzen gemittelt**; Chance- **und** Majority-Baseline im Probe-Plot; Baseline-Zuordnung über Facet-Titel (vorher konnte die Chance-Linie im falschen Facet landen).

**v2.1 (nach dem ersten Lauf):** Der Test-Split enthält **nur sBM-Fenster** (der Lauf zeigte `situation: 317/0`) — `run_probe` überspringt jetzt Labels mit < 2 Klassen (daran crashte Abschnitt 12), eine **Genre-Permutations-Probe** ersetzt die entfallene Situations-Negativkontrolle, und das Stimulus-Sampling kommt ohne `groupby.apply`-DeprecationWarning aus.

**v3 (Rhythmus-Zeit-Probes, Abschnitte 15–16):** Konsequenz aus dem ersten Probe-Lauf: Genre/Tempo-Accuracy 0.0 war ein **Split-Artefakt** (Musik-ID ≡ Genre im Test-Split, Details in Abschnitt 15). Neu: frameweise **Rhythmus-Probes** (Beat als Positiv-Kontrolle; Beat-Phase, lokales Tempo, Frames-bis-Beat als Integrationstargets) mit Input-, Kontext- und Positions-Baselines, **Counterfactual-Beat-Shift**, **Transition↔Beat-Alignment** als Verhaltens-Readout und **Settledness-Probe** (latente Uhr bei festem t). Alle Targets variieren innerhalb jedes Songs — der strenge Musik-ID-Split bleibt, Aliasing ist unmöglich.

**v3.1 (nach dem v3-Lauf):** Der Lauf ergab (a) *keine* linear lesbare Phase (Layer-R² ≤ 0 bei Fenster-Baseline 0.68) und (b) Settledness-AUC 0.90–0.99. Zwei Entscheidungs-Zellen dazu: **15e** Linear-vs-MLP-Gap (fehlt die Phase wirklich, oder ist sie nur nichtlinear kodiert?) und **16b** Label-Identitäts-Kontrolle (ist die latente Uhr echt, oder liest die Probe bei transition-lastigen Plänen nur „aktuelles Label = 0"? — Label-One-Hot-Baseline + Movement-only-Auswertung). Zelle 16 speichert dafür jetzt zusätzlich y_t und den finalen Plan.

Alle Ergebnisse (CSV/NPZ/PNG) landen dauerhaft unter `MyDrive/AtomicDance/mechinterp/`. Laufzeit: **T4 reicht völlig** (19-Mio-Parameter-Modell).

## 1) Runtime-Check

In [ ]:
import shutil, subprocess, sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
else:
    print("Keine GPU aktiv — Analyse laeuft auch auf CPU, nur langsamer.")

## 2) Setup: Drive, Ordner, Seeds, Pakete

In [ ]:
%pip install -q einops scikit-learn seaborn

In [ ]:
from google.colab import drive
from pathlib import Path
import os, random
import numpy as np
import torch

drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE = Path('/content/drive/MyDrive/AtomicDance')
MI = DRIVE / 'mechinterp'
DATA_DIR, ACT_DIR, OUT_DIR = MI / 'data', MI / 'activations', MI / 'outputs'
for p in [DATA_DIR, ACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', DEVICE)
print('Outputs →', MI)

## 3) Repo klonen & Testdaten aus dem Drive-Cache

Geklont wird **dein Fork** (`Erikiss/AtomicDance`, Stand: identisch mit `oceanflowlab/AtomicDance`, Commit `d790dbc`) — so bleibt der Lauf reproduzierbar, auch wenn sich das Original-Repo später ändert.

In [ ]:
%cd /content
!rm -rf /content/AtomicDance
!git clone -q https://github.com/Erikiss/AtomicDance.git
%cd /content/AtomicDance

In [ ]:
import glob, os, shutil, zipfile, tarfile

archives = sorted(glob.glob(str(DRIVE / 'dataset' / '*')))
assert archives, 'Kein Datensatz-Archiv im Drive-Cache (MyDrive/AtomicDance/dataset).'
src = archives[0]
dst = '/content/AtomicDance/data/atomic_aistpp'
if not os.path.exists(os.path.join(dst, 'manifest.json')):
    tmp = '/content/_atomic_extract'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    (zipfile.ZipFile(src).extractall(tmp) if src.endswith('.zip')
     else tarfile.open(src).extractall(tmp))
    hit = next(r for r, d, f in os.walk(tmp) if 'manifest.json' in f)
    os.makedirs('/content/AtomicDance/data', exist_ok=True)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.move(hit, dst)
print('Datensatz bereit:', dst)

test_music = np.load(os.path.join(dst, 'test', 'music.npy'))
test_labels = np.load(os.path.join(dst, 'test', 'labels.npy'))
import json as _json
with open(os.path.join(dst, 'test', 'names.json')) as f:
    test_names = _json.load(f)
print('music:', test_music.shape, '| labels:', test_labels.shape, '| names:', len(test_names))
print('Beispielname:', test_names[0])

## 4) Checkpoint-Inspektion (nur lesend)

Neuesten Planner-Checkpoint von Drive laden und hineinschauen: Keys, Trainings-Args, Parameterzahl. **Erwartung:** 19.121.965 State-Dict-Elemente = 19.044.965 trainierbare Parameter (Trainingslog) + 77.000 Buffer (`pe [150, 1, 512]` = 76.800, `alphas`/`alpha_bars` = 200).

In [ ]:
ckpts = sorted((DRIVE / 'runs' / 'atomic_planner').glob('*.pt'), key=os.path.getmtime)
assert ckpts, 'Kein Planner-Checkpoint auf Drive gefunden.'
CKPT_PATH = ckpts[-1]
print('Checkpoint:', CKPT_PATH)

try:
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location='cpu')

print('Top-Level-Keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

def find_state_dict(obj):
    if isinstance(obj, dict):
        for k in ['model', 'model_state', 'model_state_dict', 'state_dict', 'ema', 'ema_model']:
            if k in obj and isinstance(obj[k], dict):
                return obj[k], k
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj, '<root>'
    raise RuntimeError('State-Dict im Checkpoint nicht identifiziert — Keys oben pruefen.')

STATE, state_key = find_state_dict(ckpt)
STATE = { (k[7:] if k.startswith('module.') else k): v for k, v in STATE.items() }
n_params = sum(v.numel() for v in STATE.values() if torch.is_tensor(v))
print(f'State-Dict unter "{state_key}" | Tensoren: {len(STATE)} | Elemente: {n_params:,}')

CKPT_ARGS = {}
for k in ['args', 'config', 'hparams']:
    if isinstance(ckpt, dict) and k in ckpt:
        a = ckpt[k]
        CKPT_ARGS = dict(vars(a)) if hasattr(a, '__dict__') else dict(a)
        break
print('Trainings-Args:', CKPT_ARGS if CKPT_ARGS else '(keine im Checkpoint — nutze README-Defaults)')
print('\nErste State-Keys:', list(STATE.keys())[:8])

## 5) Planner rekonstruieren & Gewichte laden (verifiziert, v2)

Keine Discovery mehr nötig — `model/atomic_planner.py` definiert:

- `AtomicPlannerTransformer(num_atomic_classes, music_dim, latent_dim=512, num_layers=8, num_heads=8, ff_size=1024, dropout=0.1, max_seq_len=18000)` mit intern **`num_classes = num_atomic_classes + 1`**
- `UniformD3PM(model, num_steps=100, schedule='cosine')`

Die Konstruktion unten spiegelt exakt `train_atomic.py` und nutzt die im Checkpoint gespeicherten Trainings-Args (README-Defaults als Fallback). `schedule` ist egal — die Buffer werden vom Checkpoint überschrieben.

In [ ]:
import sys
import torch.nn as nn

sys.path.insert(0, '/content/AtomicDance')
from model.atomic_planner import AtomicPlannerTransformer, UniformD3PM

# Verifiziert: Checkpoint label_embedding [101,512] => num_atomic_classes=100.
# max_seq_len=150 zwingend (Default 18000 gaebe pe-Buffer [18000,1,512] -> Size-Mismatch).
PLANNER = UniformD3PM(
    AtomicPlannerTransformer(
        num_atomic_classes=int(CKPT_ARGS.get('num_classes', 100)),
        music_dim=int(CKPT_ARGS.get('music_dim', 35)),
        latent_dim=int(CKPT_ARGS.get('latent_dim', 512)),
        num_layers=int(CKPT_ARGS.get('num_layers', 8)),
        num_heads=int(CKPT_ARGS.get('num_heads', 8)),
        ff_size=int(CKPT_ARGS.get('ff_size', 1024)),
        dropout=float(CKPT_ARGS.get('dropout', 0.1)),
        max_seq_len=int(CKPT_ARGS.get('seq_len', 150)),
    ),
    num_steps=int(CKPT_ARGS.get('diffusion_steps', 100)),
)
PLANNER.load_state_dict(STATE, strict=True)
PLANNER.to(DEVICE).eval()

n_train = sum(p.numel() for p in PLANNER.parameters())
n_all = sum(v.numel() for v in PLANNER.state_dict().values())
print(f'✔ UniformD3PM(AtomicPlannerTransformer), K={PLANNER.num_classes - 1}')
print(f'  trainierbare Parameter: {n_train:,} (Soll: 19.044.965)')
print(f'  State-Dict inkl. Buffer: {n_all:,} (Soll: 19.121.965)')

## 6) Forward-Check

Die Signatur ist aus dem Repo bekannt: `forward(noisy_labels, music_features, timesteps, padding_mask=None)`. Standard-Analysezustand: `y` = uniformes Rauschen (fester Seed), `t = T_MAX − 1` — das Modell sieht dann effektiv nur die Musik, ideal für Musik-Probes.

In [ ]:
import inspect

CORE = PLANNER.model
print('Gehookter Kern:', type(CORE).__name__)
print('forward-Signatur:', inspect.signature(CORE.forward))

NUM_CLASSES = CORE.num_classes            # 101: Label 0 = Transition, 1-100 = Movements
SEQ_LEN = test_music.shape[1]             # 150
T_MAX = int(PLANNER.num_steps)            # 100

g = torch.Generator().manual_seed(SEED)
NOISE_Y = torch.randint(0, NUM_CLASSES, (1, SEQ_LEN), generator=g)

def core_forward(music_batch, y=None, t=None):
    B = music_batch.shape[0]
    y = (NOISE_Y.repeat(B, 1) if y is None else y).to(DEVICE)
    t = torch.full((B,), T_MAX - 1 if t is None else t, dtype=torch.long, device=DEVICE)
    m = music_batch.to(DEVICE).float()
    return CORE(y, m, t)  # forward(noisy_labels, music_features, timesteps, padding_mask=None)

with torch.no_grad():
    logits = core_forward(torch.tensor(test_music[:2]))
print('Logits-Shape:', tuple(logits.shape), '(erwartet: (2, 150, 101))')
assert logits.shape == (2, SEQ_LEN, NUM_CLASSES), 'Unerwartete Ausgabeform — bitte melden.'

## 7) Stimulus-Suite: Faktoren aus den Sequenznamen

Aus jedem Testfenster-Namen (`gBR_sBM_cAll_d04_mBR0_ch01_slice0`) parsen wir: **Genre** (10 Klassen), **Situation** (Basic/Advanced), **Tänzer**, **Musikstück** und die **Tempoklasse** (letzte Ziffer der Musik-ID; in der AIST-DB entspricht 0–5 aufsteigendem BPM von ca. 80–130).

In [ ]:
import pandas as pd
import re

def parse_name(name):
    m = {
        'genre': re.search(r'g([A-Z]{2})', name),
        'situation': re.search(r'_s([A-Z]{2})_', name),
        'dancer': re.search(r'_d(\d+)_', name),
        'music': re.search(r'_m([A-Z]{2}\d)', name),
        'choreo': re.search(r'_ch(\d+)', name),
    }
    row = {k: (v.group(1) if v else 'NA') for k, v in m.items()}
    row['tempo_class'] = int(row['music'][-1]) if row['music'] != 'NA' and row['music'][-1].isdigit() else -1
    row['base_seq'] = re.sub(r'(_slice.*|_win.*|_\d+)$', '', name)
    return row

meta = pd.DataFrame([{'idx': i, 'name': n, **parse_name(n)} for i, n in enumerate(test_names)])
parse_ok = (meta['genre'] != 'NA').mean()
print(f'Parse-Rate: {parse_ok:.1%} | Fenster: {len(meta)}')
print(meta['genre'].value_counts().to_dict())
print('Tempoklassen:', sorted(meta['tempo_class'].unique()))

N_STIM = min(512, len(meta))
per_genre = max(1, N_STIM // meta['genre'].nunique())
sel = meta.groupby('genre')['idx'].apply(lambda s: s.sample(min(len(s), per_genre), random_state=SEED))
stim = meta[meta['idx'].isin(sel.tolist())].reset_index(drop=True)
stim.to_csv(DATA_DIR / 'planner_stimulus_suite.csv', index=False)
print('Stimulus-Suite:', len(stim), '→', DATA_DIR / 'planner_stimulus_suite.csv')
stim.head()

## 8) Layer-Stack auswählen

Der Kern ist ein `nn.TransformerEncoder` mit 8 identischen Layern und **`batch_first=True`** — Layer-Outputs sind `[B, 150, 512]`. Wir hooken Anfang / Viertel / Mitte / Dreiviertel / Ende. (Die `enable_nested_tensor`-UserWarning kommt von `norm_first=True` und ist harmlos.)

In [ ]:
LAYERS = CORE.encoder.layers   # nn.ModuleList, 8x TransformerEncoderLayer (batch_first=True)
n_layers = len(LAYERS)
SELECTED_LAYERS = sorted(set([0, n_layers // 4, n_layers // 2, (3 * n_layers) // 4, n_layers - 1]))
print(f'Layer: {n_layers} | ausgewaehlt: {SELECTED_LAYERS} | Layer-Klasse: {type(LAYERS[0]).__name__}')

## 9) Activation-Recorder

Context-Manager mit Forward-Hooks. Musik ist kein kausaler Text — es gibt kein sinnvolles „letztes Token", deshalb **Mean-Pooling über die 150 Frames** (optional `pool='none'` für die volle `[150, d]`-Karte, z. B. für spätere Beat-Analysen).

In [ ]:
class ActivationRecorder:
    def __init__(self, layers, layer_indices, pool='mean'):
        self.layers, self.layer_indices, self.pool = layers, list(layer_indices), pool
        self.cache, self.handles = {}, []

    def _extract(self, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        if h.dim() == 3 and h.shape[0] == SEQ_LEN and h.shape[1] != SEQ_LEN:
            h = h.transpose(0, 1)  # [T,B,D] -> [B,T,D]; greift hier nie (batch_first=True)
        return h

    def _hook(self, idx):
        def fn(module, inputs, output):
            h = self._extract(output).detach().float()
            self.cache[idx] = (h.mean(dim=1) if self.pool == 'mean' else h).cpu()
        return fn

    def __enter__(self):
        self.handles = [self.layers[i].register_forward_hook(self._hook(i)) for i in self.layer_indices]
        return self

    def __exit__(self, *a):
        for h in self.handles: h.remove()

@torch.no_grad()
def capture(music_batch, layer_indices=None, pool='mean', y=None, t=None):
    layer_indices = SELECTED_LAYERS if layer_indices is None else layer_indices
    with ActivationRecorder(LAYERS, layer_indices, pool) as rec:
        core_forward(music_batch, y=y, t=t)
    return {i: rec.cache[i].numpy() for i in layer_indices}

test_acts = capture(torch.tensor(test_music[:2]))
{k: v.shape for k, v in test_acts.items()}

## 10) Kontrast-Analyse

Gruppen-Zentroiden statt einzelner Minimalpaare — robuster: **Genre** (gBR vs gHO), **Tempo** (Klasse 0 vs 5) und **Situation** (sBM vs sFM). Gemessen werden L2-Distanz und Cosine-Ähnlichkeit der mittleren Aktivierung je Layer. *Hinweis: In diesem Test-Split gibt es nur sBM-Fenster — der Situations-Kontrast wird automatisch übersprungen (Guard unten).*

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def group_centroids(df_a, df_b, batch=32):
    def acts_for(df):
        sums = {i: 0 for i in SELECTED_LAYERS}; n = 0
        for s in range(0, len(df), batch):
            mb = torch.tensor(test_music[df['idx'].values[s:s+batch]])
            a = capture(mb)
            for i in SELECTED_LAYERS: sums[i] = sums[i] + a[i].sum(0)
            n += len(a[SELECTED_LAYERS[0]])
        return {i: sums[i] / n for i in SELECTED_LAYERS}
    return acts_for(df_a), acts_for(df_b)

def cosine(a, b, eps=1e-9):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

CONTRASTS = [
    ('genre', 'gBR', 'gHO', stim[stim.genre == 'BR'], stim[stim.genre == 'HO']),
    ('tempo', 'langsam(0)', 'schnell(5)', stim[stim.tempo_class == 0], stim[stim.tempo_class == 5]),
    ('situation', 'sBM', 'sFM', stim[stim.situation == 'BM'], stim[stim.situation == 'FM']),
]

rows = []
for axis, la, lb, da, db in CONTRASTS:
    if len(da) < 3 or len(db) < 3:
        print(f'{axis}: zu wenige Beispiele ({len(da)}/{len(db)}) — uebersprungen'); continue
    ca, cb = group_centroids(da, db)
    for i in SELECTED_LAYERS:
        d = ca[i] - cb[i]
        rows.append({'axis': axis, 'a': la, 'b': lb, 'layer': i,
                     'l2_diff': float(np.linalg.norm(d)),
                     'cosine': cosine(ca[i], cb[i]),
                     'n_a': len(da), 'n_b': len(db)})

contrast_df = pd.DataFrame(rows)
contrast_df.to_csv(OUT_DIR / 'planner_contrast_layer_metrics.csv', index=False)
display(contrast_df)

plt.figure(figsize=(8, 4))
sns.lineplot(data=contrast_df, x='layer', y='l2_diff', hue='axis', marker='o')
plt.title('Planner: Kontrast-Staerke pro Layer (Gruppen-Zentroiden)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_contrast_l2_by_layer.png', dpi=160)
plt.show()

## 11) Aktivierungs-Datensatz für Probes (→ Drive)

In [ ]:
from tqdm.auto import tqdm

per_layer = {i: [] for i in SELECTED_LAYERS}
BATCH = 32
for s in tqdm(range(0, len(stim), BATCH)):
    mb = torch.tensor(test_music[stim['idx'].values[s:s+BATCH]])
    a = capture(mb)
    for i in SELECTED_LAYERS:
        per_layer[i].append(a[i])
ACTS = {i: np.concatenate(v).astype('float32') for i, v in per_layer.items()}

np.savez_compressed(ACT_DIR / 'planner_mean_activations.npz',
                    **{f'layer_{k}': v for k, v in ACTS.items()})
stim.to_csv(ACT_DIR / 'planner_activation_metadata.csv', index=False)
print({k: v.shape for k, v in ACTS.items()})
print('Gespeichert →', ACT_DIR)

## 12) Lineare Probes (Genre, Tempoklasse) + Permutations-Kontrolle

**v2-Fix:** Gruppiert wird nach **Musik-ID** (`music`-Spalte), nicht mehr nach `base_seq`. Der Modellinput ist bei `t = max` und fixem Noise-`y` **nur die Musik**, und derselbe Song (z. B. `mBR0`) kommt bei mehreren Tänzern/Choreos vor — `base_seq`-Gruppierung ließe identische Musikfeatures gleichzeitig in Train und Test (Leakage). Mit 6 Songs pro Genre wird der Split gröber (~1–2 ausgehaltene Songs pro Genre): verrauschter, aber ehrlich — und beantwortet die interessantere Frage, ob das Modell *Genre* kodiert oder nur *Song-Identität*.

**v2.1-Fix:** Der Test-Split enthält nur sBM-Fenster — die Situations-Probe hat damit nur eine Klasse und wird automatisch übersprungen (daran crashte der erste Lauf: `ValueError: … only one class: 'BM'`). Als **Negativkontrolle** dient stattdessen die Genre-Probe mit **zufällig permutierten Labels**: erwartet ist Chance-Niveau; deutlich mehr ⇒ Artefakt in der Auswertungs-Pipeline. Im Plot: gestrichelt rot = 1/K-Chance, gepunktet grau = Majority-Baseline im Test-Split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def run_probe(label_col, y_override=None, tag=None):
    tag = tag or label_col
    mask = stim[label_col].astype(str) != '-1'
    y = (stim.loc[mask, label_col].astype(str).values if y_override is None
         else np.asarray(y_override)[mask.values])
    # v2.1-Guard: Labels mit nur einer Klasse (z. B. situation: Test-Split ist rein sBM) ueberspringen.
    if len(np.unique(y)) < 2:
        print(f'{tag}: nur eine Klasse ({np.unique(y)[0]}) — uebersprungen')
        return None
    # v2: Gruppierung nach Musik-ID — derselbe Song darf nie in Train UND Test liegen.
    groups = stim.loc[mask, 'music'].values
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                  .split(np.zeros(len(y)), y, groups))
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        print(f'{tag}: degenerierter Group-Split (Train/Test einklassig) — uebersprungen')
        return None
    _, counts = np.unique(y[te], return_counts=True)
    majority = counts.max() / counts.sum()
    recs = []
    for i in SELECTED_LAYERS:
        X = ACTS[i][mask.values]
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
        clf.fit(X[tr], y[tr])
        recs.append({'label': tag, 'layer': i,
                     'accuracy': accuracy_score(y[te], clf.predict(X[te])),
                     'chance': 1.0 / len(np.unique(y)),
                     'majority': majority,
                     'n_test': len(te)})
    return pd.DataFrame(recs)

# Negativkontrolle: Genre-Labels permutieren — bricht die X-y-Kopplung, erwartet: Chance-Niveau.
genre_shuffled = np.random.RandomState(SEED).permutation(stim['genre'].astype(str).values)

probes = [run_probe('genre'), run_probe('tempo_class'), run_probe('situation'),
          run_probe('genre', y_override=genre_shuffled, tag='genre_shuffled')]
probe_df = pd.concat([p for p in probes if p is not None], ignore_index=True)
probe_df.to_csv(OUT_DIR / 'planner_linear_probes.csv', index=False)
display(probe_df)

In [ ]:
g = sns.catplot(data=probe_df, x='layer', y='accuracy', col='label', kind='bar',
                color='#4c78a8', height=3.4, aspect=1.0)
for ax in g.axes.flat:
    lbl = ax.get_title().split(' = ')[-1]
    sub = probe_df[probe_df['label'] == lbl]
    ax.axhline(sub['chance'].iloc[0], ls='--', c='red', lw=1)
    ax.axhline(sub['majority'].iloc[0], ls=':', c='gray', lw=1)
    ax.set_ylim(0, 1)
g.figure.suptitle('Lineare Probes im Residual Stream des Planners', y=1.04)
g.figure.savefig(OUT_DIR / 'planner_linear_probes.png', dpi=160, bbox_inches='tight')
plt.show()

## 13) D3PM-Spezialität: Wann legen sich Entscheidungen fest?

Ein Pre-Hook auf dem Denoiser-Kern protokolliert bei jedem Reverse-Schritt den Label-Zustand `y_t` — der Sampler ruft den Kern genau **1× pro Schritt** mit `y_t` als `[B, 150]`-Long-Tensor auf (verifiziert in `UniformD3PM.sample`). Gemessen wird pro Schritt der Anteil der Frames, die schon dem finalen Plan entsprechen — getrennt für **Transitions** (Label 0) und **Movements** (1–100). Legt sich zuerst die Struktur fest oder zuerst der Inhalt?

**Interpretations-Hinweis:** `sample()` zieht in jedem Schritt *alle* Positionen neu aus `Categorical(logits)` — nichts ist je hart fixiert; die Kurve misst, wie früh die Logits scharf werden (effektive Kristallisation). Für reproduzierbare Rollouts: `deterministic=True`. **v2:** Mittel über mehrere Sequenzen (graue Linien = Einzelsequenzen).

In [ ]:
print('Sampler:', inspect.signature(PLANNER.sample))

traj_states = []

def pre_hook(module, args, kwargs):
    for v in list(args) + list(kwargs.values()):
        if torch.is_tensor(v) and v.dtype == torch.long and v.dim() == 2 and v.shape[-1] == SEQ_LEN:
            traj_states.append(v[0].detach().cpu().clone())
            break

def record_trajectory(idx, deterministic=False):
    traj_states.clear()
    music_1 = torch.tensor(test_music[idx:idx+1]).to(DEVICE).float()
    h = CORE.register_forward_pre_hook(pre_hook, with_kwargs=True)
    try:
        with torch.no_grad():
            plan = PLANNER.sample(music_1, deterministic=deterministic)
    finally:
        h.remove()
    final = plan[0].detach().cpu()
    rows = []
    for step, y in enumerate(traj_states):
        match = (y == final)
        rows.append({'idx': idx, 'step': step,
                     'match_all': match.float().mean().item(),
                     'match_transitions': match[final == 0].float().mean().item() if (final == 0).any() else np.nan,
                     'match_movements': match[final > 0].float().mean().item() if (final > 0).any() else np.nan})
    return pd.DataFrame(rows), final

In [ ]:
N_TRAJ = 16   # v2: ueber mehrere Sequenzen mitteln statt nur einer
trajs = []
for idx in stim['idx'].head(N_TRAJ):
    df_i, final = record_trajectory(int(idx))
    trajs.append(df_i)
traj_df = pd.concat(trajs, ignore_index=True)
traj_df.to_csv(OUT_DIR / 'planner_denoising_trajectory.csv', index=False)
print(f'{traj_df["idx"].nunique()} Sequenzen | {traj_df["step"].max() + 1} Reverse-Schritte')

mean_df = traj_df.groupby('step').mean(numeric_only=True).reset_index()
plt.figure(figsize=(8, 4))
for _, sub in traj_df.groupby('idx'):
    plt.plot(sub['step'], sub['match_all'], color='gray', alpha=0.15, lw=0.7)
for col, lbl in [('match_all', 'alle Frames'), ('match_transitions', 'Transitions (Label 0)'),
                 ('match_movements', 'Movements (1-100)')]:
    plt.plot(mean_df['step'], mean_df[col], label=lbl, lw=2)
plt.xlabel('Reverse-Diffusionsschritt'); plt.ylabel('Anteil = finaler Plan')
plt.title(f'Entscheidungs-Kristallisation im D3PM (Mittel ueber {N_TRAJ} Sequenzen)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_denoising_trajectory.png', dpi=160)
plt.show()

## 14) Kausal-Skelett: Music-Swap & Aktivierungs-Patching

(a) **Music-Swap** als Verhaltensbaseline: Plan mit Musik A vs. Musik B (anderes Genre) — wie stark ändern sich Transition-Anteil und Movement-Verteilung? (b) **Aktivierungs-Patch** (Skelett, nach den Probes mit dem besten Layer füllen): Layer-Outputs sind `[B, 150, 512]` (`batch_first=True`), `dim=1` ist also die **Zeitachse** — der Mean-Shift im Skelett ist dimensionsrichtig.

In [ ]:
def plan_stats(music_tensor, deterministic=False):
    with torch.no_grad():
        p = PLANNER.sample(music_tensor.to(DEVICE).float(), deterministic=deterministic)
    p = p[0].detach().cpu()
    segs = int((p[1:] != p[:-1]).sum()) + 1
    return {'transition_frac': float((p == 0).float().mean()),
            'n_segments': segs,
            'top_moves': torch.bincount(p[p > 0], minlength=NUM_CLASSES).topk(3).indices.tolist()}

a_row = stim[stim.genre == 'BR'].iloc[0]
b_row = stim[stim.genre == 'HO'].iloc[0]
m_a = torch.tensor(test_music[int(a_row['idx']):int(a_row['idx'])+1])
m_b = torch.tensor(test_music[int(b_row['idx']):int(b_row['idx'])+1])
print('Musik A (gBR):', plan_stats(m_a))
print('Musik B (gHO):', plan_stats(m_b))

In [ ]:
# --- Aktivierungs-Patch-Skelett (nach den Probes mit dem besten Layer fuellen) ---
# Layer-Outputs sind [B, 150, 512] (batch_first=True) -> dim=1 ist die Zeitachse.
# BEST_LAYER = 4
# source_centroid = torch.tensor(ACTS[BEST_LAYER][(stim.genre == 'BR').values].mean(0))
# def patch_hook(module, inputs, output):
#     h = output[0] if isinstance(output, (tuple, list)) else output
#     h = h + (source_centroid.to(h.device, h.dtype) - h.mean(dim=1, keepdim=True))
#     return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
# handle = LAYERS[BEST_LAYER].register_forward_hook(patch_hook)
# try:
#     print('gepatcht:', plan_stats(m_b))
# finally:
#     handle.remove()

## 15) Rhythmus-Zeit-Probes (v3)

Antwort auf den 0.0-Befund aus Abschnitt 12. Der war ein **Struktur-Artefakt**: In diesem Test-Split hat jedes Genre genau einen Song, Musik-ID ≡ Genre — der Group-Split hält also ganze Genres aus dem Training, und eine im Training fehlende Klasse ist für die Probe *prinzipiell* unvorhersagbar (deshalb exakt 0.0 statt Chance). Keine Aussage über den Residual Stream.

Der Ausweg (JoruriPuppet-Prinzip „Beat-Tabelle"): **zeitvariable Targets**, die innerhalb jedes Songs variieren — der strenge Musik-ID-Split bleibt, Aliasing ist unmöglich. Targets aus der Beat-Spur der Musikfeatures (Layout verifiziert in `data/audio_extraction/baseline_features.py`: `[Envelope | 20×MFCC | 12×Chroma | Peak | Beat]` → Beat = Dim 34):

- **beat** (binär je Frame) — *Positiv-Kontrolle*: steht direkt im Input, muss überall lesbar sein.
- **phase** (Position zwischen zwei Beats, als sin/cos-Regression) — steht **nicht** im Input; der Input hat nur Impulse an Beat-Frames. Phase an einem Nicht-Beat-Frame erfordert zeitliche Integration durch Attention.
- **ibi** (lokales Tempo = Inter-Beat-Intervall in Frames) und **to_next_beat** — ebenfalls Integrationstargets.

Baselines, die die Aktivierungs-Probes schlagen müssen: rohe Musikframes (35 dim), ±8-Frame-Kontextfenster (595 dim; kann Phase *lokal* teilweise ableiten), Positions-One-Hot (150 dim; fängt „Phase-aus-Fensterposition-memorieren" ab). Erst *Aktivierung ≫ alle Baselines* belegt Integration durch das **Modell** statt Durchleitung.

In [ ]:
FPS = 60   # AIST++-Features laufen mit 60 fps; die Targets unten sind fps-agnostisch (Frames)
ENV_DIM, PEAK_DIM, BEAT_DIM = 0, 33, 34   # Layout: [env | mfcc20 | chroma12 | peak | beat]

bd = test_music[:, :, BEAT_DIM]
thr = 0.5 * (bd.max() + bd.min())   # robust, falls die Spur skaliert sein sollte
beat_grid = bd > thr
if len(np.unique(bd)) > 2:
    print(f'Hinweis: Beat-Spur nicht binaer ({len(np.unique(bd))} Werte) — Schwellwert = {thr:.3f}')
assert beat_grid.any() and not beat_grid.all(), 'Beat-Spur (Dim 34) unplausibel — Feature-Layout pruefen!'

n_beats_win = beat_grid[stim['idx'].values].sum(1)
print(f'Beats pro Fenster: min {int(n_beats_win.min())} / median {int(np.median(n_beats_win))} / max {int(n_beats_win.max())}')

def rhythm_targets(beats):
    T = len(beats); idx = np.flatnonzero(beats)
    phase = np.full(T, np.nan); ibi = np.full(T, np.nan); to_next = np.full(T, np.nan)
    for a, b in zip(idx[:-1], idx[1:]):
        f = np.arange(a, b)
        phase[f] = (f - a) / (b - a); ibi[f] = b - a; to_next[f] = b - f
    return phase, ibi, to_next

tt = [rhythm_targets(b) for b in beat_grid[stim['idx'].values]]
PHASE = np.stack([t[0] for t in tt]); IBI = np.stack([t[1] for t in tt]); TONEXT = np.stack([t[2] for t in tt])
VALID = ~np.isnan(PHASE)
print(f'Valide Frames: {VALID.mean():.1%} | medianes IBI: {np.nanmedian(IBI):.0f} Frames '
      f'(~{60 * FPS / np.nanmedian(IBI):.0f} BPM bei {FPS} fps)')

w0 = 0
fig, ax = plt.subplots(figsize=(9, 2.6))
env = test_music[stim['idx'].iloc[w0], :, ENV_DIM]
ax.plot(env / (np.abs(env).max() + 1e-9), c='gray', alpha=0.6, label='Envelope (norm.)')
ax.vlines(np.flatnonzero(beat_grid[stim['idx'].iloc[w0]]), 0, 1, color='red', lw=1, label='Beat')
ax.plot(PHASE[w0], c='#4c78a8', lw=1.2, label='Phase-Target')
ax.set_xlabel('Frame'); ax.set_title(f'Rhythmus-Targets: {stim["name"].iloc[w0]}')
ax.legend(loc='upper right', fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, r2_score

# Frame-Aktivierungen [Fenster, 150, 512] pro Layer (Musik-only-Zustand wie in Abschnitt 6)
FRAME_LAYERS = SELECTED_LAYERS
frame_acts = {i: [] for i in FRAME_LAYERS}
for s in tqdm(range(0, len(stim), BATCH)):
    mb = torch.tensor(test_music[stim['idx'].values[s:s+BATCH]])
    a = capture(mb, pool='none')
    for i in FRAME_LAYERS:
        frame_acts[i].append(a[i].astype('float32'))
FACTS = {i: np.concatenate(v) for i, v in frame_acts.items()}
print('Frame-Aktivierungen:', {i: v.shape for i, v in FACTS.items()})

RAW = test_music[stim['idx'].values].astype('float32')
K = 8
pad = np.pad(RAW, ((0, 0), (K, K), (0, 0)))
WIN = np.concatenate([pad[:, k:k+SEQ_LEN] for k in range(2 * K + 1)], axis=-1)
POS = np.broadcast_to(np.eye(SEQ_LEN, dtype='float32'), (len(stim), SEQ_LEN, SEQ_LEN)).copy()

groups_w = stim['music'].values
tr_w, te_w = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                  .split(np.zeros(len(stim)), groups=groups_w))
print(f'Split: {len(tr_w)} Train- / {len(te_w)} Test-Fenster | Test-Songs: {sorted(set(groups_w[te_w]))}')

def flat_masked(X3, Y2, V2, rows):
    X = X3[rows].reshape(-1, X3.shape[-1])
    y = Y2[rows].ravel(); v = V2[rows].ravel()
    return X[v], y[v]

def ridge_score(X3, Y2, V2, metric):
    Xtr, ytr = flat_masked(X3, Y2, V2, tr_w)
    Xte, yte = flat_masked(X3, Y2, V2, te_w)
    sc = StandardScaler().fit(Xtr)
    m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr)
    pred = m.predict(sc.transform(Xte))
    return float(roc_auc_score(yte, pred)) if metric == 'auc' else float(r2_score(yte, pred))

BEAT_Y = beat_grid[stim['idx'].values].astype('float32')
ALL_V = np.ones_like(VALID)
FEATURE_SETS = {f'layer_{i}': FACTS[i] for i in FRAME_LAYERS}
FEATURE_SETS.update({'input_raw35': RAW, 'input_window17x35': WIN, 'pos_onehot': POS})

rows_out = []
for fname, X3 in FEATURE_SETS.items():
    rows_out.append({'features': fname, 'target': 'beat', 'metric': 'auc',
                     'score': ridge_score(X3, BEAT_Y, ALL_V, 'auc')})
    r2s = ridge_score(X3, np.sin(2 * np.pi * PHASE), VALID, 'r2')
    r2c = ridge_score(X3, np.cos(2 * np.pi * PHASE), VALID, 'r2')
    rows_out.append({'features': fname, 'target': 'phase', 'metric': 'r2', 'score': (r2s + r2c) / 2})
    rows_out.append({'features': fname, 'target': 'ibi', 'metric': 'r2',
                     'score': ridge_score(X3, IBI, VALID, 'r2')})
    rows_out.append({'features': fname, 'target': 'to_next_beat', 'metric': 'r2',
                     'score': ridge_score(X3, TONEXT, VALID, 'r2')})
    print(fname, 'fertig')

frame_probe_df = pd.DataFrame(rows_out)
frame_probe_df.to_csv(OUT_DIR / 'planner_rhythm_frame_probes.csv', index=False)
display(frame_probe_df.pivot(index='features', columns='target', values='score').round(3))

def get(feat, tgt):
    m = frame_probe_df[(frame_probe_df['features'] == feat) & (frame_probe_df['target'] == tgt)]
    return m['score'].iloc[0]

lay = sorted(FRAME_LAYERS)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for tgt, mk in [('phase', 'o'), ('ibi', 's'), ('to_next_beat', '^')]:
    axes[0].plot(lay, [get(f'layer_{i}', tgt) for i in lay], marker=mk, label=tgt)
for base, ls in [('input_raw35', ':'), ('input_window17x35', '--'), ('pos_onehot', '-.')]:
    axes[0].axhline(get(base, 'phase'), ls=ls, c='gray', lw=1, label=f'phase-Baseline: {base}')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('R² (gehaltene Songs)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3); axes[0].set_title('Rhythmus-Integration pro Layer')
axes[1].plot(lay, [get(f'layer_{i}', 'beat') for i in lay], marker='o', c='#4c78a8', label='Aktivierungen')
for base, ls in [('input_raw35', ':'), ('input_window17x35', '--'), ('pos_onehot', '-.')]:
    axes[1].axhline(get(base, 'beat'), ls=ls, c='gray', lw=1, label=base)
axes[1].axhline(0.5, c='red', lw=1, alpha=0.5, label='Chance')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('ROC-AUC'); axes[1].set_ylim(0.4, 1.02)
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3); axes[1].set_title('Beat-Detektion (Positiv-Kontrolle)')
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_rhythm_probes.png', dpi=160); plt.show()

### 15c) Counterfactual: Beat-Spur verschieben

Peak- und Beat-Dim werden um ein halbes medianes IBI zirkulär gerollt, alle anderen 33 Dims bleiben identisch. Die auf **clean** trainierte Phase-Probe wird auf den CF-Aktivierungen gegen (a) die *verschobenen* und (b) die *originalen* Phase-Targets getestet. Das trennt die Quelle des Signals: folgt die Probe der expliziten Beat-Spur (Dim 33/34) oder den impliziten Rhythmus-Korrelaten in Envelope/MFCC/Chroma?

In [ ]:
SHIFT = int(round(np.nanmedian(IBI) / 2))
print(f'CF-Shift: {SHIFT} Frames (halbes medianes IBI)')

CF_RAW = RAW.copy()
CF_RAW[:, :, [PEAK_DIM, BEAT_DIM]] = np.roll(CF_RAW[:, :, [PEAK_DIM, BEAT_DIM]], SHIFT, axis=1)

cf_beats = np.roll(beat_grid[stim['idx'].values], SHIFT, axis=1)
cf_tt = [rhythm_targets(b) for b in cf_beats]
CF_PHASE = np.stack([t[0] for t in cf_tt])
V_JOINT = VALID & ~np.isnan(CF_PHASE)

cf_facts = {i: [] for i in FRAME_LAYERS}
for s in range(0, len(stim), BATCH):
    a = capture(torch.tensor(CF_RAW[s:s+BATCH]), pool='none')
    for i in FRAME_LAYERS:
        cf_facts[i].append(a[i].astype('float32'))
CF_FACTS = {i: np.concatenate(v) for i, v in cf_facts.items()}

def phase_r2(models, X3, PH, rows):
    sc, msin, mcos = models
    X, ys = flat_masked(X3, np.sin(2 * np.pi * PH), V_JOINT, rows)
    _, yc = flat_masked(X3, np.cos(2 * np.pi * PH), V_JOINT, rows)
    Xs = sc.transform(X)
    return (r2_score(ys, msin.predict(Xs)) + r2_score(yc, mcos.predict(Xs))) / 2

cf_out = []
for i in FRAME_LAYERS:
    Xtr, ys_tr = flat_masked(FACTS[i], np.sin(2 * np.pi * PHASE), V_JOINT, tr_w)
    _, yc_tr = flat_masked(FACTS[i], np.cos(2 * np.pi * PHASE), V_JOINT, tr_w)
    sc = StandardScaler().fit(Xtr)
    models = (sc,
              Ridge(alpha=10.0).fit(sc.transform(Xtr), ys_tr),
              Ridge(alpha=10.0).fit(sc.transform(Xtr), yc_tr))
    cf_out.append({'layer': i,
                   'clean_r2': phase_r2(models, FACTS[i], PHASE, te_w),
                   'cf_vs_shifted_r2': phase_r2(models, CF_FACTS[i], CF_PHASE, te_w),
                   'cf_vs_original_r2': phase_r2(models, CF_FACTS[i], PHASE, te_w)})
cf_df = pd.DataFrame(cf_out)
cf_df.to_csv(OUT_DIR / 'planner_phase_cf_transfer.csv', index=False)
display(cf_df.round(3))
print('Lesart: cf_vs_shifted >> cf_vs_original -> Phase-Signal folgt der expliziten Beat-Spur.')
print('        cf_vs_original >> cf_vs_shifted -> Phase-Signal stammt aus Envelope/MFCC/Chroma.')

### 15d) Verhaltens-Check: Rasten Transitions auf Beats ein?

Segmentgrenzen der gesampelten Pläne (`deterministic=True`) gegen Beat-Frames; Kontrolle über zufällige zirkuläre Beat-Shifts (**q** = Anteil der Zufalls-Shifts mit schlechterem Alignment: q≈1 stark aligniert, q≈0.5 Zufall). Unter CF-Musik zeigt der Vergleich „alte vs. verschobene Beats", ob auch das *Verhalten* der expliziten Beat-Spur folgt.

In [ ]:
def transition_beat_alignment(plan, beats, n_shift=200, rng=None):
    p = np.asarray(plan)
    bounds = np.flatnonzero(p[1:] != p[:-1]) + 1
    bidx = np.flatnonzero(beats)
    if len(bounds) == 0 or len(bidx) == 0:
        return np.nan, np.nan
    d = np.abs(bounds[:, None] - bidx[None, :]).min(1).mean()
    rng = rng if rng is not None else np.random.RandomState(SEED)
    null = []
    for _ in range(n_shift):
        sb = np.flatnonzero(np.roll(beats, rng.randint(1, len(beats))))
        null.append(np.abs(bounds[:, None] - sb[None, :]).min(1).mean())
    return float(d), float(np.mean(d < np.array(null)))

N_ALIGN = 24
rng = np.random.RandomState(SEED)
align_rows = []
for r in range(min(N_ALIGN, len(stim))):
    with torch.no_grad():
        p_clean = PLANNER.sample(torch.tensor(RAW[r:r+1]).to(DEVICE), deterministic=True)[0].cpu().numpy()
        p_cf = PLANNER.sample(torch.tensor(CF_RAW[r:r+1]).to(DEVICE), deterministic=True)[0].cpu().numpy()
    b_clean = beat_grid[stim['idx'].iloc[r]]
    b_cf = np.roll(b_clean, SHIFT)
    d_cc, q_cc = transition_beat_alignment(p_clean, b_clean, rng=rng)
    d_fo, _ = transition_beat_alignment(p_cf, b_clean, rng=rng)
    d_fs, q_fs = transition_beat_alignment(p_cf, b_cf, rng=rng)
    align_rows.append({'name': stim['name'].iloc[r], 'd_clean': d_cc, 'q_clean': q_cc,
                       'd_cf_vs_alt': d_fo, 'd_cf_vs_verschoben': d_fs, 'q_cf_verschoben': q_fs})
align_df = pd.DataFrame(align_rows)
align_df.to_csv(OUT_DIR / 'planner_beat_alignment.csv', index=False)
print(align_df[['d_clean', 'd_cf_vs_alt', 'd_cf_vs_verschoben']].mean().round(2))
print(f'q (Anteil Zufalls-Shifts schlechter): clean {align_df["q_clean"].mean():.2f} | '
      f'CF vs. verschobene Beats {align_df["q_cf_verschoben"].mean():.2f}')
display(align_df.head(8))

### 15e) Linear-vs-MLP-Gap für die Phase (v3.1)

Der v3-Lauf zeigte: Phase ist **linear** nicht dekodierbar (Layer-R² ≤ 0), während das ±8-Input-Fenster 0.68 erreicht. Offene Frage: fehlt die Phase im Stream *wirklich*, oder ist sie nur **nichtlinear kodiert**? Test nach dem Notebook-02-Prinzip: dieselbe Probe als kleines MLP (1 Hidden-Layer, 128 Units, Subsample für CPU-Laufzeit; ~5–10 min). `mlp_r2(layer) ≫ linear_r2(layer)` ⇒ Phase existiert nichtlinear (Finetuning aufs *Lesbarmachen* unnötig). Beide ≈ ≤ 0 bei `mlp_r2(window) > 0` ⇒ Phase fehlt im Stream tatsächlich.

In [ ]:
from sklearn.neural_network import MLPRegressor

MLP_SETS = ['layer_0', 'layer_4', 'layer_7', 'input_window17x35']
N_SUB = 12000   # Subsample fuer CPU-Laufzeit

def mlp_phase_score(X3):
    Xtr, ys_tr = flat_masked(X3, np.sin(2 * np.pi * PHASE), VALID, tr_w)
    _, yc_tr = flat_masked(X3, np.cos(2 * np.pi * PHASE), VALID, tr_w)
    Xte, ys_te = flat_masked(X3, np.sin(2 * np.pi * PHASE), VALID, te_w)
    _, yc_te = flat_masked(X3, np.cos(2 * np.pi * PHASE), VALID, te_w)
    sub = np.random.RandomState(SEED).permutation(len(Xtr))[:N_SUB]
    sc = StandardScaler().fit(Xtr[sub])
    out = []
    for ytr, yte in [(ys_tr, ys_te), (yc_tr, yc_te)]:
        mlp = MLPRegressor(hidden_layer_sizes=(128,), max_iter=150, early_stopping=True,
                           n_iter_no_change=5, random_state=SEED)
        mlp.fit(sc.transform(Xtr[sub]), ytr[sub])
        out.append(r2_score(yte, mlp.predict(sc.transform(Xte))))
    return float(np.mean(out))

gap_rows = []
for fname in MLP_SETS:
    gap_rows.append({'features': fname,
                     'linear_r2': get(fname, 'phase'),
                     'mlp_r2': mlp_phase_score(FEATURE_SETS[fname])})
    print(fname, 'fertig')
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT_DIR / 'planner_phase_linear_vs_mlp.csv', index=False)
display(gap_df.round(3))
print('Lesart: mlp_r2(layer) >> linear_r2(layer)  -> Phase existiert, aber nichtlinear kodiert.')
print('        beide <= 0 bei mlp_r2(window) > 0  -> Phase fehlt im Stream wirklich.')

## 16) Latente Uhr: Settledness bei festem Diffusionsschritt (v3)

Analogon zu den τ_dlm-Probes aus dem Subliminal-Clocks-Setup — mit einem Twist: Der Diffusionsschritt t ist beim Planner **expliziter Input** (Time-Embedding), eine t-Probe wäre zirkulär. Nicht-trivial ist die frameweise **Settledness**: „Ist dieses Frame schon final?" variiert *bei festem t* von Sample zu Sample und Frame zu Frame — das Time-Embedding ist je Schritt konstant, jede dekodierbare Settledness muss aus dem y_t-Inhalt kommen. AUC deutlich über 0.5 (bei Basisrate ≠ 0/1) heißt: Der Stream führt eine frameweise **Verifikations-Spur**. (Die Probe darf dabei das aktuelle Label-Embedding lesen — genau das ist die Frage: ist *Stabilität* linear markiert?)

In [ ]:
N_TAU = 48
TAU_LAYER = SELECTED_LAYERS[len(SELECTED_LAYERS) // 2]
TAU_STRIDE = 10

tau_stim = stim.sample(N_TAU, random_state=SEED).reset_index(drop=True)

def sample_with_acts(idx):
    states, acts, call = [], [], {'i': 0}
    def y_pre(module, args, kwargs):
        for v in list(args) + list(kwargs.values()):
            if torch.is_tensor(v) and v.dtype == torch.long and v.dim() == 2 and v.shape[-1] == SEQ_LEN:
                if call['i'] % TAU_STRIDE == 0:
                    states.append(v[0].detach().cpu().clone())
                break
    def a_fwd(module, inputs, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        if call['i'] % TAU_STRIDE == 0:
            acts.append(h[0].detach().float().cpu().numpy())
        call['i'] += 1
    h1 = CORE.register_forward_pre_hook(y_pre, with_kwargs=True)
    h2 = LAYERS[TAU_LAYER].register_forward_hook(a_fwd)
    try:
        with torch.no_grad():
            plan = PLANNER.sample(torch.tensor(test_music[idx:idx+1]).to(DEVICE).float())
    finally:
        h1.remove(); h2.remove()
    steps = list(range(0, call['i'], TAU_STRIDE))
    return steps, states, acts, plan[0].detach().cpu()

per_step, fins = {}, {}
for r in tqdm(range(N_TAU)):
    steps, states, acts, fin = sample_with_acts(int(tau_stim['idx'].iloc[r]))
    fins[r] = fin.numpy()
    for si, y, h in zip(steps, states, acts):
        per_step.setdefault(si, []).append((h, y.numpy(), (y == fin).numpy(), r))

mus_tau = tau_stim['music'].values
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
                  .split(np.zeros(N_TAU), groups=mus_tau))

tau_rows = []
for si in sorted(per_step):
    H = np.stack([p[0] for p in per_step[si]])
    S = np.stack([p[2] for p in per_step[si]])
    R = np.array([p[3] for p in per_step[si]])
    tr_m = np.isin(R, tr_i); te_m = np.isin(R, te_i)
    Xtr, ytr = H[tr_m].reshape(-1, H.shape[-1]), S[tr_m].ravel()
    Xte, yte = H[te_m].reshape(-1, H.shape[-1]), S[te_m].ravel()
    row = {'step': si, 't': T_MAX - 1 - si, 'settled_rate': float(S.mean()), 'auc': np.nan}
    if min(ytr.sum(), len(ytr) - ytr.sum()) >= 50 and min(yte.sum(), len(yte) - yte.sum()) >= 50:
        sc = StandardScaler().fit(Xtr)
        m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr.astype('float32'))
        row['auc'] = float(roc_auc_score(yte, m.predict(sc.transform(Xte))))
    tau_rows.append(row)
tau_df = pd.DataFrame(tau_rows)
tau_df.to_csv(OUT_DIR / 'planner_settledness_probe.csv', index=False)
display(tau_df.round(3))

plt.figure(figsize=(7, 3.5))
plt.plot(tau_df['step'], tau_df['auc'], marker='o', label=f'Settled-Probe AUC (Layer {TAU_LAYER})')
plt.plot(tau_df['step'], tau_df['settled_rate'], marker='.', ls='--', c='gray', label='Settled-Anteil (Basisrate)')
plt.axhline(0.5, c='red', lw=1, alpha=0.5)
plt.xlabel('Reverse-Schritt (t = 99 − Schritt)'); plt.ylim(0, 1.02)
plt.title('Latente Uhr: Ist „Frame schon final?" linear lesbar?')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_settledness_probe.png', dpi=160)
plt.show()

### 16b) Kontrolle: Verifikations-Spur oder nur Label-Identität? (v3.1)

Der v3-Lauf zeigte AUC 0.90–0.99 — aber die gesampelten Pläne sind **transition-lastig** (Abschnitt 14: `transition_frac` 0.8–1.0). Wenn der finale Plan überwiegend Label 0 ist, gilt „settled" ≈ „aktuelles Label ist 0", und das kann eine Probe *trivial* aus dem Label-Embedding lesen — ganz ohne echtes Stabilitäts-Signal. Zwei Kontrollen:

1. **Label-One-Hot-Baseline:** dieselbe Settled-Probe, aber die Features sind nur One-Hot(aktuelles Label y_t) (101-dim). Alles, was diese Baseline erreicht, ist Label-Identität, keine Verifikation.
2. **Movement-only-Auswertung:** AUC nur auf Frames mit finalem Label > 0 — dort hilft „Label ist 0" nicht mehr.

**Echt** ist die Verifikations-Spur nur, wenn `auc_act` die Label-Baseline in *beiden* Auswertungen deutlich schlägt.

In [ ]:
eye = np.eye(NUM_CLASSES, dtype='float32')

def auc_for(X4, S, tr_m, te_m, eval_mask):
    Xtr = X4[tr_m].reshape(-1, X4.shape[-1]); ytr = S[tr_m].ravel()
    Xte = X4[te_m].reshape(-1, X4.shape[-1]); yte = S[te_m].ravel()
    em = eval_mask[te_m].ravel()
    Xte, yte = Xte[em], yte[em]
    if min(ytr.sum(), len(ytr) - ytr.sum()) < 50 or min(yte.sum(), len(yte) - yte.sum()) < 50:
        return np.nan
    sc = StandardScaler().fit(Xtr)
    m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr.astype('float32'))
    return float(roc_auc_score(yte, m.predict(sc.transform(Xte))))

rows16b = []
for si in sorted(per_step):
    H = np.stack([p[0] for p in per_step[si]])
    Y = np.stack([p[1] for p in per_step[si]])
    S = np.stack([p[2] for p in per_step[si]])
    R = np.array([p[3] for p in per_step[si]])
    F = np.stack([fins[r] for r in R])
    tr_m = np.isin(R, tr_i); te_m = np.isin(R, te_i)
    X_lab = eye[Y]
    all_mask = np.ones_like(S, dtype=bool)
    mov_mask = F > 0
    rows16b.append({'step': si, 't': T_MAX - 1 - si,
                    'auc_act': auc_for(H, S, tr_m, te_m, all_mask),
                    'auc_label': auc_for(X_lab, S, tr_m, te_m, all_mask),
                    'auc_act_mov': auc_for(H, S, tr_m, te_m, mov_mask),
                    'auc_label_mov': auc_for(X_lab, S, tr_m, te_m, mov_mask),
                    'settled_rate': float(S.mean()),
                    'movement_frac': float(mov_mask.mean())})
ctrl_df = pd.DataFrame(rows16b)
ctrl_df.to_csv(OUT_DIR / 'planner_settledness_controls.csv', index=False)
display(ctrl_df.round(3))

plt.figure(figsize=(8, 4))
plt.plot(ctrl_df['step'], ctrl_df['auc_act'], marker='o', label='Aktivierungen (alle Frames)')
plt.plot(ctrl_df['step'], ctrl_df['auc_label'], marker='s', ls='--', label='Label-One-Hot (alle Frames)')
plt.plot(ctrl_df['step'], ctrl_df['auc_act_mov'], marker='o', alpha=0.6, label='Aktivierungen (nur Movements)')
plt.plot(ctrl_df['step'], ctrl_df['auc_label_mov'], marker='s', ls='--', alpha=0.6, label='Label-One-Hot (nur Movements)')
plt.axhline(0.5, c='red', lw=1, alpha=0.5)
plt.xlabel('Reverse-Schritt'); plt.ylabel('ROC-AUC'); plt.ylim(0.35, 1.02)
plt.title('Settledness: echte Verifikations-Spur vs. Label-Identitaet')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_settledness_controls.png', dpi=160)
plt.show()

d_all = (ctrl_df['auc_act'] - ctrl_df['auc_label']).mean()
d_mov = (ctrl_df['auc_act_mov'] - ctrl_df['auc_label_mov']).mean()
print(f'mittlerer Vorsprung Aktivierungen vs. Label-Baseline: alle Frames {d_all:+.3f} | nur Movements {d_mov:+.3f}')
print('Lesart: beide Vorspruenge deutlich > 0  -> echte frameweise Verifikations-Spur.')
print('        Vorsprung ~ 0                   -> die "Uhr" war nur Label-Identitaet.')

## Ausblick

- **CoAx-artige Head-Ablationen** (Analogon Notebook 04): Damage = Phase-Probe-Fehler + Transition-Beat-Alignment → „Beat-Heads" prinzipiiert identifizieren statt raten.
- **Phase-Steering** (Analogon Notebook 02): Phase-Richtung aus der Probe in Layer L addieren bzw. rollen — verschieben sich die Transitions im gesampelten Plan?
- **Genre-Generalisierung** ist mit diesem Test-Split strukturell unmessbar (1 Song pro Genre). Wer sie will, zieht Probing-Fenster aus `data/atomic_aistpp/train` (mehrere Songs pro Genre) — der Planner bleibt dabei eingefroren.
- **Music-Swap-Matrix** Genre×Genre mit `deterministic=True`; **Noise-Robustheit** der Mean-Probes über mehrere `NOISE_Y`-Seeds.

Alle Artefakte liegen unter `MyDrive/AtomicDance/mechinterp/` und überleben die Session.